Broad cross-asset example list you can try in `assets` configs:

- iShares Core S&P 500 (`CSSPX.MI`)
- iShares Core MSCI World (`SWDA.MI`)
- iShares Core MSCI Emerging Markets IMI (`EIMI.MI`)
- iShares Nasdaq 100 (`CSNDX.MI`)
- iShares MSCI ACWI (`IUSQ.MI`)
- Vanguard FTSE All-World (`VWCE.MI`)
- iShares Core DAX (`EXS1.MI`)
- Lyxor Core STOXX Europe 600 (DR) (`MEUD.MI`)
- iShares Core MSCI Europe (`SMEA.MI`)
- Xtrackers MSCI USA (`XD9U.MI`)
- Xtrackers MSCI Emerging Markets (`XMME.MI`)
- iShares Core EURO STOXX 50 (`CSSX5E.MI`)
- Real Estate Sector (`XLRE`)
- ETF Bond USA 7 (`GOVT`)
- ETF Bond USA 10 (`TLH`)
- ETF Bond USA 15 (`LTPZ`)
- ETF Bond USA 20 (`XTLT.TO`)
- ETF Bond USA 30 (`UTHY`)
- ETF Inflation Adjusted USA 7 (`TIP`)
- ETF Inflation Adjusted USA 15 (`LTPZ`)
- Bitcoin (`BTC=F`)
- Gold (`GC=F`)
- Silver (`SI=F`)
- Crude Oil (`CL=F`)
- Cash Liquidity (`CSH.PA`)


In [9]:
#asset defintion 
from orchestration import *


asset_info = [
        SyntheticLETFAssetConfig(
            id="SWDA_2X",
            underlying_ticker="SWDA.MI",
            leverage=2.0,
            ter=0.0092,
            spread=0.0030,
        ),
        SpotAssetConfig(
            id="GOVIES",
            ticker="GOVT",
        ),
    ]

In [10]:
import importlib

import data_loader
import orchestration

importlib.reload(data_loader)
importlib.reload(orchestration)
from orchestration import *

# Rebuild assets with refreshed dataclass types after module reload.
normalized_asset_info = []
for asset in asset_info:
    if hasattr(asset, "underlying_ticker"):
        normalized_asset_info.append(
            SyntheticLETFAssetConfig(
                id=asset.id,
                underlying_ticker=asset.underlying_ticker,
                leverage=float(asset.leverage),
                ter=float(asset.ter),
                spread=float(getattr(asset, "spread", 0.0)),
            )
        )
    elif hasattr(asset, "ticker"):
        normalized_asset_info.append(
            SpotAssetConfig(
                id=asset.id,
                ticker=asset.ticker,
            )
        )
    else:
        raise ValueError(f"Unsupported asset entry: {asset}")

asset_info = normalized_asset_info

# Define the simulation configuration 
# with market data, assets, portfolio settings, and Monte Carlo parameters.
config = SimulationConfig(
    market=MarketDataConfig(
        start="2018-04-03",
        end="2025-12-31",
        fred_series="SOFR",
        fred_is_percent=True,
    ),
    assets=asset_info,
    portfolio=PortfolioConfig(
        target_weights={
            "SWDA_2X": 0.80,
            "GOVIES": 0.20,
        },
        initial_capital=100_000.0,
        rebalance_frequency_days=252,
        tolerance_band=0.05,
        capital_gains_tax_rate=0.26,
    ),
    monte_carlo=MonteCarloConfig(
        n_paths=5000,
        horizon_days=2520,
        method="bootstrap",
        distribution="normal",
        student_t_df=6.0,
        seed=42,
    ),
    metrics_ruin_threshold_fraction=0.10,
    use_mean_risk_free_for_metrics=True,
)

result = run_complete_simulation(config)

print("Wealth paths shape:", result.portfolio.wealth_paths.shape)
print("Simulated returns shape:", result.simulated_asset_returns.shape)
print("Metrics summary:")
display(result.metrics.summary)


Base tickers needed for historical data:
{'GOVT', 'SWDA.MI'}



1 Failed download:
['GOVT']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


TypeError: '>=' not supported between instances of 'numpy.ndarray' and 'Timestamp'

## Visual Diagnostics

Run this cell after the simulation cell to inspect path behavior and tail outcomes.

In [ ]:
import importlib

import numpy as np
import pandas as pd

import visuals
from orchestration import SpotAssetConfig, SyntheticLETFAssetConfig

importlib.reload(visuals)

from visuals import (
    plot_drawdown_chart,
    plot_spaghetti_paths,
    plot_terminal_wealth_distribution,
)

wealth_paths = result.portfolio.wealth_paths
terminal = wealth_paths[:, -1]

terminal_summary = pd.Series(
    {
        "min": float(np.min(terminal)),
        "p5": float(np.quantile(terminal, 0.05)),
        "median": float(np.median(terminal)),
        "mean": float(np.mean(terminal)),
        "p95": float(np.quantile(terminal, 0.95)),
        "max": float(np.max(terminal)),
    },
    name="TerminalWealth",
)

display(terminal_summary.to_frame())

asset_lines = []
for asset in config.assets:
    weight = float(config.portfolio.target_weights.get(asset.id, 0.0))

    if isinstance(asset, SpotAssetConfig):
        asset_lines.append(
            f"{asset.ticker}| weight={weight:.1%}"
        )
    elif isinstance(asset, SyntheticLETFAssetConfig):
        asset_lines.append(
            f"{asset.underlying_ticker} {asset.leverage:.1f}x | weight={weight:.1%} | TER={asset.ter:.2%} | spread={asset.spread:.2%}"
        )

assets_subtitle ="<br>".join(asset_lines)

summary_note = (
    f"FRED={config.market.fred_series}"
    "<br>"
    f"{config.monte_carlo.n_paths:,} draws | {config.monte_carlo.horizon_days} horizon days"
    "<br>"
    f"{config.monte_carlo.method} Method<br>Distribution={config.monte_carlo.distribution}<br>Student_t_df={config.monte_carlo.student_t_df}"
    "<br>"

    f"Initial capital={config.portfolio.initial_capital:,.2f}<br>Rebalancing Frequency={config.portfolio.rebalance_frequency_days} days"
    "<br>"
    f"Tolerance band={config.portfolio.tolerance_band:.2%}<br>Capital Gain Tax={config.portfolio.capital_gains_tax_rate:.2%}"
    "<br>"
)

fig_spaghetti = plot_spaghetti_paths(
    wealth_paths=wealth_paths,
    n_sample=100,
    seed=42,
    normalize_to_1=False,
    title="Monte Carlo Spaghetti",
    subtitle=assets_subtitle,
    bottom_note=summary_note,
    subtitle_align="left",
    bottom_note_align="left",
    bottom_note_x=0.65,
    bottom_note_y=-0.10,
    bottom_note_box=True,
    backend="plotly",
    width=800,
    height=600,
)

metrics_summary_raw = (
    result.metrics.summary.to_string()
    if hasattr(result.metrics.summary, "to_string")
    else str(result.metrics.summary)
)
metrics_summary_html = metrics_summary_raw.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
metrics_summary_html = metrics_summary_html.replace(" ", "&nbsp;").replace("\n", "<br>")

fig_spaghetti.add_annotation(
    x=0.0,
    y=-0.1,
    xref="paper",
    yref="paper",
    text=f"<br><span style='font-family:monospace'>{metrics_summary_html}</span>",
    showarrow=False,
    align="left",
    xanchor="left",
    yanchor="top",
    font={"size": 10, "color": "#0f172a"},
    bgcolor="rgba(255, 255, 255, 0.82)",
    bordercolor="#cbd5e1",
    borderwidth=1,
)

fig_terminal = plot_terminal_wealth_distribution(
    wealth_paths=wealth_paths,
    bins=60,
    title="Distribution of Terminal Wealth",
    backend="plotly",
    width=800,
    height=600,
)

terminal_footnote = (
    f"Footnote: min={terminal_summary['min']:,.0f} | p5={terminal_summary['p5']:,.0f} | "
    f"median={terminal_summary['median']:,.0f} | mean={terminal_summary['mean']:,.0f} | "
    f"p95={terminal_summary['p95']:,.0f} | max={terminal_summary['max']:,.0f}"
)

fig_terminal.update_layout(margin={"t": 80, "b": 120, "l": 70, "r": 30})
fig_terminal.add_annotation(
    x=0.1,
    y=-0.10,
    xref="paper",
    yref="paper",
    text=terminal_footnote,
    showarrow=False,
    align="left",
    xanchor="left",
    yanchor="top",
    font={"size": 11, "color": "#334155"},
)

 
fig_drawdown = plot_drawdown_chart(
    wealth_paths=wealth_paths,
    drawdowns=result.metrics.drawdowns,
    title="Drawdown Chart: Median vs Worst Case",
    backend="plotly",
    width=1200,
    height=720,
)

fig_spaghetti.show()
fig_terminal.show()
#fig_drawdown.show()


NameError: name 'result' is not defined